# Phase 1 — Test: lib/ Skeleton + show_source + should_skip (v2)

**v2:** `show_source` rendert jetzt standardmässig als **Markdown** —
JupyterLab nutzt sein eigenes Syntax-Highlighting, das zum aktiven Theme passt.
Kein Kontrastproblem mehr bei hellem Theme.

Dieses Notebook prüft, dass:

1. Das `lib/`-Package korrekt importierbar ist
2. `show_source(...)` im **Markdown-Modus** aufklappbare Quellcode-Blöcke erzeugt (default)
3. `show_source(..., mode='html', style=...)` alternativ pygments nutzt
4. `should_skip(...)` korrekt entscheidet

**Voraussetzung:** Dieses Notebook liegt in `test/`, also eine Ebene unter dem Projekt-Root.
Falls du es auf Projekt-Root-Ebene legst, ändere unten `_lib_root = Path('.').resolve()`.

In [ ]:
# ── lib-Import Setup ─────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Dieses Notebook liegt in test/ — ein Ebene hoch zu Projekt-Root:
_lib_root = Path('..').resolve()
if str(_lib_root) not in sys.path:
    sys.path.insert(0, str(_lib_root))

%load_ext autoreload
%autoreload 2

print(f'sys.path enthält: {_lib_root}')
print(f'lib/ existiert:   {(_lib_root / "lib").exists()}')


## Test 1 — Imports funktionieren


In [ ]:
from lib.plotting import show_source, should_skip
from lib import __version__ as lib_version

print(f'✅ lib/ Version {lib_version} geladen')
print(f'   show_source:  {show_source}')
print(f'   should_skip:  {should_skip}')


## Test 2 — `show_source()` Default-Modus (Markdown)

Unter dieser Zelle sollte ein **zugeklappter** Block erscheinen:  
🔎 *Quellcode: `should_skip` (aus `lib/plotting.py`)*  

Beim Klick öffnet sich ein Codeblock mit JupyterLab's nativem Syntax-
Highlighting — passt automatisch zum aktiven Theme (hell/dunkel).

In [ ]:
show_source(should_skip)


## Test 3 — `collapsed=False` — offen statt aufgeklappt


In [ ]:
show_source(show_source, collapsed=False)


## Test 4 — HTML-Fallback mit verschiedenen pygments-Styles

Falls das Markdown-Rendering in deinem Setup nicht funktioniert (ältere
nbviewer-Versionen, Moodle-Preview, HTML-Export), kannst du `mode='html'`
nutzen. Dann kommt pygments mit fest gewähltem Style zum Einsatz.

Verfügbare Style-Empfehlungen:

- **Hell:** `default`, `friendly`, `tango`, `sas`
- **Dunkel:** `monokai`, `one-dark`, `github-dark`, `dracula` (falls installiert)

Vergleiche unten die Lesbarkeit:

In [ ]:
# Hell: 'default' (sehr lesbar auf weissem Jupyter-HG)
show_source(should_skip, mode='html', style='default',
            title='HTML + pygments style="default" (hell)')


In [ ]:
# Hell: 'friendly' — weicher, pastell-artig
show_source(should_skip, mode='html', style='friendly',
            title='HTML + pygments style="friendly" (hell, weich)')


In [ ]:
# Dunkel: 'monokai' — klassisch, hoher Kontrast
show_source(should_skip, mode='html', style='monokai',
            title='HTML + pygments style="monokai" (dunkel, hoher Kontrast)')


In [ ]:
# Dunkel: 'one-dark' — wie VS Code
show_source(should_skip, mode='html', style='one-dark',
            title='HTML + pygments style="one-dark" (VS Code)')


### Alle verfügbaren pygments-Styles auflisten


In [ ]:
from pygments.styles import get_all_styles
print('Verfügbare pygments-Styles:')
for s in sorted(get_all_styles()):
    print(f'  {s}')


## Test 5 — `should_skip()` Entscheidungs-Matrix

Prüfung aller relevanten Pfade:

| Fall | out_path | asset_type | name | overrides | Erwartung |
|---|---|---|---|---|---|
| 1 | existiert | animation | — | — | `True` (skip) |
| 2 | fehlt | animation | — | — | `False` (rendern) |
| 3 | existiert | statisch | — | — | `False` (default always) |
| 4 | existiert | statisch | match override skip | skip | `True` (skip) |
| 5 | existiert | animation | match override force | force | `False` (rendern) |


In [ ]:
import os, tempfile

_tmp = tempfile.mkdtemp()
_exists = os.path.join(_tmp, 'exists.gif')
_missing = os.path.join(_tmp, 'missing.gif')
open(_exists, 'w').write('dummy')

CFG = {
    'animation': {
        'modus':          'skip_if_exists',
        'modus_statisch': 'always',
        'overrides': {
            'my_expensive_map': 'skip_if_exists',
            'my_dev_chart':     'force_rebuild',
        }
    }
}

tests = [
    (_exists,  'animation', 'some_anim',         True,  'Fall 1: anim exists, default skip'),
    (_missing, 'animation', 'some_anim',         False, 'Fall 2: anim fehlt'),
    (_exists,  'statisch',  'some_chart',        False, 'Fall 3: stat exists, default always'),
    (_exists,  'statisch',  'my_expensive_map',  True,  'Fall 4: stat exists, override skip'),
    (_exists,  'animation', 'my_dev_chart',      False, 'Fall 5: anim exists, override force'),
]

for out, typ, name, expected, desc in tests:
    got = should_skip(out, typ, name, CFG)
    status = '✅' if got == expected else '❌'
    print(f'{status} {desc:<40s} got={got} expected={expected}')


## Test 6 — Master-Schalter `skip_if_exists_all`

Setzt man `CFG['animation']['modus_statisch'] = 'skip_if_exists_all'`, werden
**alle** statischen Charts auf `skip_if_exists` umgeschaltet.

In [ ]:
CFG_master = {
    'animation': {
        'modus': 'skip_if_exists',
        'modus_statisch': 'skip_if_exists_all',
    }
}

got = should_skip(_exists, 'statisch', 'any_chart_name', CFG_master)
print(f'{"✅" if got else "❌"} Master-Schalter skip_if_exists_all → {got}')


## Test 7 — Robustheit gegen fehlerhafte Inputs


In [ ]:
import warnings

# Falsch getippter Modus → Warning + Fallback
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    got = should_skip(_exists, 'animation', 'x', {'animation': {'modus': 'typo'}})
    print(f'{"✅" if got == False and len(w) == 1 else "❌"} Typo im Modus → {len(w)} Warning(s), Ergebnis {got}')

# asset_type falsch → ValueError
try:
    should_skip(_exists, 'video', 'x', CFG)
    print('❌ Hätte ValueError werfen sollen')
except ValueError as e:
    print(f'✅ ValueError bei asset_type=video: {e}')

# show_source mit ungültigem mode → ValueError
try:
    show_source(should_skip, mode='xml')
    print('❌ Hätte ValueError werfen sollen')
except ValueError as e:
    print(f'✅ ValueError bei mode=xml: {e}')


## Abschluss

Wenn alle Tests ✅ zeigen und Test 2 einen aufklappbaren Codeblock mit
korrekt lesbarem Syntax-Highlighting (passend zu deinem JupyterLab-Theme)
produziert, ist **Phase 1** erfolgreich.

### Empfehlung

- **In den Projekt-Notebooks** immer `show_source(fn)` (default markdown) nutzen
  — bleibt theme-passend, bester Kontrast
- **`mode='html'` nur wenn nötig**, z.B. für HTML-Export oder Moodle-Preview
- **Style-Parameter erst relevant** wenn HTML-Mode + Theme-Unabhängigkeit gebraucht werden

### Nächster Schritt

Phase 2 (Anker-Hygiene) — mechanische Bereinigung von ID-Ankern mit
führenden Ziffern in 8 Notebooks.
